In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv('../dataGenerated/gurgaon_properties_post_feature_selection_v2.csv')

In [4]:
df['luxury_category'].value_counts()

luxury_category
Low       2156
Medium    1417
Name: count, dtype: int64

In [5]:
df.shape

(3573, 14)

In [161]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,flat,sector 7,0.45,2,2,1,West,1000.0,0,Relatively New,Unfurnished,4.00,Low,Mid Floor
1,flat,sector 3,0.50,2,2,1,West,722.0,0,Old Property,Furnished,4.25,Low,Low Floor
2,flat,sohna road,0.40,2,2,3,East,661.0,0,New Property,Unfurnished,4.25,Low,High Floor
3,flat,sector 61,1.47,2,2,2,East,1333.0,0,Relatively New,Unfurnished,4.20,Low,Low Floor
4,flat,sector 92,0.70,2,2,3,East,1217.0,0,Under Construction,Unfurnished,4.00,Low,Mid Floor


In [162]:
df.shape

(3573, 14)

In [163]:
X=df.drop(columns=['price'])
y=df['price']

In [164]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

Ordinal Encoding

In [165]:
df.head(1)

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,flat,sector 7,0.45,2,2,1,West,1000.0,0,Relatively New,Unfurnished,4.0,Low,Mid Floor


In [166]:
columns_to_encode = ['property_type','sector', 'balcony', 'age_category', 'facing','furnishing_type', 'luxury_category', 'floor_category']

In [167]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'combined_rating']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [168]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [169]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [170]:
scores.mean(),scores.std()

(np.float64(0.7231722894196498), np.float64(0.026636618297866323))

In [171]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [172]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [173]:
y_pred=pipeline.predict(X_test)

In [174]:
y_pred = np.expm1(y_pred)

In [175]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.8516557201405093

In [176]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [177]:
# Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Support Vector Machine
from sklearn.svm import SVR

# Tree Models
from sklearn.tree import DecisionTreeRegressor

# Ensemble Models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

# Neural Network
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

In [178]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [179]:
# model_output = []
# for model_name,model in model_dict.items():
#     model_output.append(scorer(model_name, model))

In [180]:
# model_output

In [181]:
# model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [182]:
# model_df.sort_values(['mae'])

OneHotEncoding

In [183]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'combined_rating']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first'),['sector','age_category','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [184]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [185]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [186]:
scores.mean(),scores.std()

(np.float64(0.8328602361409331), np.float64(0.0236694543877276))

In [187]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [188]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [189]:
y_pred = pipeline.predict(X_test)

In [190]:
y_pred = np.expm1(y_pred)
mean_absolute_error(np.expm1(y_test),y_pred)

0.6529537887043967

In [191]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [192]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [193]:
# model_output = []
# for model_name,model in model_dict.items():
#     model_output.append(scorer(model_name, model))

In [194]:
# model_df=pd.DataFrame(model_output,columns=['model','r2','mae'])
# model_df.sort_values(['mae'])

OneHotEncoding With PCA

In [195]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'combined_rating']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','age_category'])
    ], 
    remainder='passthrough'
)

In [196]:
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('pca',PCA(n_components=0.95)),
    ('regressor',LinearRegression())
])

In [197]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [198]:
scores.mean(),scores.std()

(np.float64(0.05627286062255317), np.float64(0.02802707257448993))

In [199]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [200]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [201]:
# model_output = []
# for model_name,model in model_dict.items():
#     model_output.append(scorer(model_name, model))
# model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
# model_df.sort_values(['mae'])

Target Encoder

In [202]:
# !pip install category_encoders

In [203]:
df.head(1)

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,flat,sector 7,0.45,2,2,1,West,1000.0,0,Relatively New,Unfurnished,4.0,Low,Mid Floor


In [204]:
import category_encoders as ce

columns_to_encode = ['property_type', 'balcony', 'facing','furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room','combined_rating']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['age_category']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [205]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [206]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [207]:
scores.mean(),scores.std()

(np.float64(0.8046943256273483), np.float64(0.020980754913835747))

In [208]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [209]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [210]:
# model_output = []
# for model_name,model in model_dict.items():
#     model_output.append(scorer(model_name, model))
# model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
# model_df.sort_values(['mae'])

Hyperparameter Tuning

In [211]:
# !pip install optuna

In [212]:
import optuna
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor

In [213]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

ordinal_cols = [
    'age_category',
    'furnishing_type',
    'luxury_category',
    'floor_category'
]

onehot_cols = [
    'property_type',
    'balcony',
    'facing'
]

target_cols = ['sector']

preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', OrdinalEncoder(), ordinal_cols),
        ('onehot', OneHotEncoder(drop='first',
                                 handle_unknown='ignore',
                                 sparse_output=False), onehot_cols),
        ('target', ce.TargetEncoder(), target_cols)
    ],
    remainder='passthrough'
)

In [214]:
def objective(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "max_depth": trial.suggest_int("max_depth", 3, 12),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            15
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            10
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.5,
            20
        ),

        "max_bin": trial.suggest_categorical(
            "max_bin",
            [128,256,512]
        ),

        "grow_policy": trial.suggest_categorical(
            "grow_policy",
            ["depthwise","lossguide"]
        ),

        "tree_method":"hist",

        "objective":"reg:squarederror",

        "random_state":42,

        "n_jobs":-1
    }

    model = XGBRegressor(**params)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    score = cross_val_score(
        pipeline,
        X,
        y_transformed,
        scoring="r2",
        cv=cv,
        n_jobs=-1
    ).mean()

    return score

In [215]:
# study = optuna.create_study(direction="maximize")

# study.optimize(
#     objective,
#     n_trials=100,
#     show_progress_bar=True
# )

In [216]:
# print("Best R² :", study.best_value)
# print()

# print("Best Parameters")

# for k,v in study.best_params.items():
#     print(k,":",v)

In [217]:
# best_model = XGBRegressor(

#     **study.best_params,

#     objective="reg:squarederror",

#     tree_method="hist",

#     random_state=42,

#     n_jobs=-1
# )

# pipeline = Pipeline([
#     ("preprocessor", preprocessor),
#     ("model", best_model)
# ])

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y_transformed,
#     test_size=0.2,
#     random_state=42
# )

# pipeline.fit(X_train,y_train)

# pred = pipeline.predict(X_test)

# pred = np.expm1(pred)

# mae = mean_absolute_error(
#     np.expm1(y_test),
#     pred
# )

# r2 = pipeline.score(X_test,y_test)

# print("MAE :",mae)
# print("R²  :",r2)

In [218]:
# xgboost with random and gridsearchcv 

In [219]:
import category_encoders as ce

columns_to_encode = ['property_type', 'balcony', 'facing','furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room','combined_rating']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['age_category']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [220]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [221]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [222]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        tree_method='hist'
    ))
])

In [223]:
random_params = {

    'model__n_estimators': randint(300,1800),

    'model__max_depth': randint(3,12),

    'model__learning_rate': uniform(0.01,0.29),

    'model__subsample': uniform(0.6,0.4),

    'model__colsample_bytree': uniform(0.6,0.4),

    'model__min_child_weight': randint(1,12),

    'model__gamma': uniform(0,5),

    'model__reg_alpha': uniform(0,5),

    'model__reg_lambda': uniform(0.5,10)
}

In [224]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=random_params,
    n_iter=10,
    scoring='r2',
    cv=5,
    verbose=3,
    n_jobs=-1,
    random_state=42,
    error_score='raise'   # <-- important
)

# random_search.fit(X, y_transformed)

In [225]:
# print(random_search.best_score_)

# print(random_search.best_params_)

In [226]:
# best = random_search.best_params_

# grid_params = {

#     'model__max_depth': [
#         max(2, best['model__max_depth'] - 1),
#         best['model__max_depth'],
#         best['model__max_depth'] + 1
#     ],

#     'model__learning_rate': [
#         max(0.01, best['model__learning_rate'] - 0.05),
#         best['model__learning_rate'],
#         min(0.5, best['model__learning_rate'] + 0.05)
#     ],

#     'model__n_estimators': [
#         max(200, best['model__n_estimators'] - 200),
#         best['model__n_estimators'],
#         best['model__n_estimators'] + 200
#     ],

#     'model__min_child_weight': [
#         max(1, best['model__min_child_weight'] - 2),
#         best['model__min_child_weight'],
#         best['model__min_child_weight'] + 2
#     ],

#     'model__subsample': [
#         max(0.6, best['model__subsample'] - 0.05),
#         best['model__subsample'],
#         min(1.0, best['model__subsample'] + 0.05)
#     ],

#     'model__colsample_bytree': [
#         max(0.6, best['model__colsample_bytree'] - 0.05),
#         best['model__colsample_bytree'],
#         min(1.0, best['model__colsample_bytree'] + 0.05)
#     ]
# }

In [227]:
# grid_search = GridSearchCV(
#     estimator=pipeline,
#     param_grid=grid_params,
#     scoring='r2',
#     cv=5,
#     n_jobs=-1,
#     verbose=2
# )

# grid_search.fit(X, y_transformed)

# print(grid_search.best_score_)
# print(grid_search.best_params_)

In [228]:
# from sklearn.model_selection import train_test_split
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_absolute_error, r2_score
# from xgboost import XGBRegressor
# import numpy as np

# # Best parameters from GridSearchCV
# best_params = {
#     'colsample_bytree': 0.7801997007878172,
#     'learning_rate': 0.23323850914860728,
#     'max_depth': 4,
#     'min_child_weight': 10,
#     'n_estimators': 876,
#     'subsample': 0.7464101864104047
# }

# # Create final model
# final_model = XGBRegressor(
#     objective='reg:squarederror',
#     random_state=42,
#     tree_method='hist',
#     n_jobs=-1,
#     **best_params
# )

# # Pipeline
# pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', final_model)
# ])

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y_transformed,
#     test_size=0.2,
#     random_state=42
# )

# # Train
# pipeline.fit(X_train, y_train)

# # Predict
# y_pred_log = pipeline.predict(X_test)

# # Convert back to original scale
# y_pred = np.expm1(y_pred_log)
# y_true = np.expm1(y_test)

# # Metrics
# mae = mean_absolute_error(y_true, y_pred)
# r2 = r2_score(y_true, y_pred)

# print("MAE :", mae)
# print("R²  :", r2)

In [229]:
# rf targeted encoding 

In [230]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['sqrt', 'log2']
}

In [231]:
columns_to_encode = ['property_type','sector', 'balcony', 'age_category', 'facing','furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'combined_rating']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['age_category']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [232]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [233]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4,error_score='raise')

In [234]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['sqrt', 'log2'], 'regressor__max_samples': [0.1, 0.25, ...], 'regressor__n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the hi

In [235]:
search.best_params_
# {'regressor__max_depth': None,
#  'regressor__max_features': 'log2',
#  'regressor__max_samples': 1.0,
#  'regressor__n_estimators': 300}

{'regressor__max_depth': None,
 'regressor__max_features': 'sqrt',
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [236]:
final_pipe = search.best_estimator_

In [237]:
search.best_score_

np.float64(0.8980805020486666)

In [238]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Best Parameters
best_params = {
    'max_depth': None,
    'max_features': 'log2',
    'max_samples': 1.0,
    'n_estimators': 300
}

# Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        **best_params
    ))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

# Train
pipeline.fit(X_train, y_train)

# Predict
y_pred_log = pipeline.predict(X_test)

# Convert back to original scale
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"MAE : {mae:.4f}")

MAE : 0.4904


In [239]:
final_pipe.fit(X,y_transformed)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [240]:
X

,property_type,sector,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,flat,sector 7,2,2,1,West,1000.0,0,Relatively New,Unfurnished,4.00,Low,Mid Floor
1,flat,sector 3,2,2,1,West,722.0,0,Old Property,Furnished,4.25,Low,Low Floor
2,flat,sohna road,2,2,3,East,661.0,0,New Property,Unfurnished,4.25,Low,High Floor
3,flat,sector 61,2,2,2,East,1333.0,0,Relatively New,Unfurnished,4.20,Low,Low Floor
4,flat,sector 92,2,2,3,East,1217.0,0,Under Construction,Unfurnished,4.00,Low,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3568,house,sector 57,3,3,3,North-West,1700.0,0,Moderately Old,Furnished,4.00,Medium,Low Floor
3569,house,sector 26,4,4,3,North-East,1800.0,1,Moderately Old,Unfurnished,5.00,Low,Low Floor
3570,house,sector 25,3,2,3,North,1350.0,0,Old Property,Unfurnished,5.00,Low,Low Floor
3571,house,sector 26,3,3,2,East,1350.0,1,Moderately Old,Unfurnished,5.00,Low,Low Floor


In [241]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [242]:
X.columns

# X.iloc[0].values

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony', 'facing',
       'built_up_area', 'servant room', 'age_category', 'furnishing_type',
       'combined_rating', 'luxury_category', 'floor_category'],
      dtype='str')

In [243]:
data = [['house', 'sohna road', 4, 3, '3+', 'East', 2750, 0,'New Property', 'Unfurnished', 4,'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony', 'facing',
       'built_up_area', 'servant room', 'age_category', 'furnishing_type',
       'combined_rating', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,house,sohna road,4,3,3+,East,2750,0,New Property,Unfurnished,4,Low,Low Floor


In [244]:
np.expm1(pipeline.predict(one_df))

array([1.89509498])

In [245]:
df[(df['sector']=='sohna road')]['price'].mean()
#  (df['bedRoom']==4)  & (df['bathroom']==4)

np.float64(0.5746480263157894)